# 12. Przygotowanie danych do analizy spektralnej

**Arkadiusz Liber** · *Falki w praktyce. Od wzoru do kodu*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/writingin/falki-w-praktyce-od-wzoru-do-kodu-notatniki/blob/main/rozdzial-12.ipynb)

Notatnik do książki. Zawiera wszystkie programy z tekstu rozdziału oraz ćwiczenia z rozwiązaniami.

Notatnik działa lokalnie (Jupyter) i w Google Colab. Komórki można uruchamiać po kolei poleceniem *Uruchom wszystko*.

In [1]:
# Uruchom tę komórkę jako pierwszą (lokalnie albo w Google Colab).
import sys, subprocess, importlib.util, os
if importlib.util.find_spec("pywt") is None:      # PyWavelets
    subprocess.check_call([sys.executable, "-m", "pip",
                           "install", "-q", "PyWavelets"])
import numpy as np
import matplotlib.pyplot as plt
import pywt

SZARY, NIEBIESKI, RDZAWY, ZIELONY = "#757575", "#1273E2", "#D93B0B", "#078832"
FIOLET, BURSZTYN = "#9258D5", "#B36000"
plt.rcParams.update({
    "text.color": SZARY, "axes.labelcolor": SZARY, "axes.edgecolor": SZARY,
    "xtick.color": SZARY, "ytick.color": SZARY, "axes.titlecolor": SZARY,
    "font.size": 10, "axes.titlesize": 10,
    "axes.spines.top": False, "axes.spines.right": False})


## Programy z tekstu rozdziału

### Kod 12.1. Interpolacja danych na siatkę o stałym kroku

In [2]:
import numpy as np
from scipy.interpolate import CubicSpline

x = np.array([1., 2, 4, 5, 7, 8, 9])
y = np.array([1., 2, 6, 4, 2, 6, 5])

x_nowe = np.arange(x[0], x[-1] + 1)            # stały krok 1
y_liniowa = np.interp(x_nowe, x, y)
y_sklejana = CubicSpline(x, y)(x_nowe)
print(x_nowe)
print(y_liniowa)
print(y_sklejana.round(2))

[1. 2. 3. 4. 5. 6. 7. 8. 9.]
[1. 2. 4. 6. 4. 3. 2. 6. 5.]
[1.   2.   4.71 6.   4.   1.58 2.   6.   5.  ]


### Kod 12.2. Zmiana liczby próbek do potęgi dwójki

In [3]:
import numpy as np
from scipy.signal import resample, resample_poly

t = np.linspace(0, 1, 1000, endpoint=False)
s = np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*40*t)

n = 2**int(np.round(np.log2(len(s))))          # potęga dwójki
s_fft = resample(s, n)                         # metoda fourierowska
s_poly = resample_poly(s, 128, 125)            # filtr wielofazowy
t_n = np.linspace(0, 1, n, endpoint=False)
wzorzec = np.sin(2*np.pi*5*t_n) + 0.5*np.sin(2*np.pi*40*t_n)
print(len(s), "→", len(s_fft), len(s_poly))
print("błąd metody fourierowskiej:",
      f"{np.abs(s_fft - wzorzec).max():.1e}")
print("błąd filtru wielofazowego (środek):",
      f"{np.abs(s_poly - wzorzec)[50:-50].max():.1e}")

1000 → 1024 1024
błąd metody fourierowskiej: 2.3e-14
błąd filtru wielofazowego (środek): 1.1e-03


### Kod 12.3. Uzupełnianie braków i usuwanie wartości odstających filtrem Hampela

In [4]:
import numpy as np

def uzupelnij_braki(y):
    """Zastępuje wartości NaN interpolacją liniową."""
    y = y.copy()
    brak = np.isnan(y)
    n = np.arange(len(y))
    y[brak] = np.interp(n[brak], n[~brak], y[~brak])
    return y

def filtr_hampela(y, okno=5, prog=3.0):
    """Zastępuje wartości odstające medianą otoczenia."""
    y = y.copy()
    zmienione = []
    for i in range(len(y)):
        otoczenie = y[max(0, i - okno):i + okno + 1]
        m = np.median(otoczenie)
        mad = np.median(np.abs(otoczenie - m))
        if np.abs(y[i] - m) > prog*1.4826*mad:
            y[i] = m
            zmienione.append(i)
    return y, zmienione

rng = np.random.default_rng(4)
t = np.linspace(0, 1, 400)
czysty = np.sin(2*np.pi*2*t) + 0.5*t
y = czysty + 0.05*rng.normal(size=t.size)
y[[60, 150, 151, 300]] += [4, -3, 5, 6]         # błędy grube
y[200:208] = np.nan                             # luka w danych

y1 = uzupelnij_braki(y)
y2, indeksy = filtr_hampela(y1)
print("zmienione próbki:", indeksy)
print("błąd przed:", round(np.abs(y1 - czysty).max(), 2),
      " po:", round(np.abs(y2 - czysty).max(), 2))

zmienione próbki: [48, 51, 60, 150, 151, 252, 300, 370]
błąd przed: 5.97  po: 0.15


### Kod 12.4. Tryby brzegowe: liczba współczynników i wielkość współczynników przy brzegu

In [5]:
import numpy as np
import pywt

print(pywt.pad(np.array([1., 2, 3, 4]), 3, "symmetric"))
print(pywt.pad(np.array([1., 2, 3, 4]), 3, "smooth"))

t = np.linspace(0, 1, 1000)
s = np.exp(1.5*t) + 0.2*np.sin(2*np.pi*4*t)     # sygnał z trendem
falka = pywt.Wavelet("db4")
print("tryb           liczba   max |cD1|   max |cD1|")
print("               wsp.     na brzegu   w środku")
for tryb in ("zero", "constant", "symmetric", "reflect",
             "periodic", "smooth", "periodization"):
    cA, cD = pywt.dwt(s, falka, mode=tryb)
    print(f"{tryb:14s} {len(cA) + len(cD):5d}   "
          f"{max(np.abs(cD[:4]).max(), np.abs(cD[-4:]).max()):9.2e}"
          f"   {np.abs(cD[8:-8]).max():9.2e}")

[3. 2. 1. 1. 2. 3. 4. 4. 3. 2.]
[-2. -1.  0.  1.  2.  3.  4.  5.  6.  7.]
tryb           liczba   max |cD1|   max |cD1|
               wsp.     na brzegu   w środku
zero            1006    2.17e+00    4.19e-08
constant        1006    2.99e-03    4.19e-08
symmetric       1006    7.61e-04    4.19e-08
reflect         1006    5.98e-03    4.19e-08
periodic        1006    1.69e+00    4.19e-08
smooth          1006    9.69e-07    4.19e-08
periodization   1000    8.05e-01    4.19e-08


### Kod 12.5. Transformata Haara w schemacie liftingu, w wersji rzeczywistej i całkowitoliczbowej

In [6]:
import numpy as np
import pywt

def haar_lifting(x):
    e, o = x[0::2], x[1::2]            # podział
    d = o - e                          # predykcja
    s = e + d/2                        # uaktualnienie
    return s, d

def haar_lifting_odwrotny(s, d):
    e = s - d/2                        # cofnięcie uaktualnienia
    o = d + e                          # cofnięcie predykcji
    x = np.empty(2*len(s), dtype=s.dtype)
    x[0::2], x[1::2] = e, o            # połączenie
    return x

x = np.array([1., 2, 2, 4, 3, 2, 1, 2, 2, 3])
s, d = haar_lifting(x)
print(s, d)
cA, cD = pywt.dwt(x, "haar")
print(np.allclose(np.sqrt(2)*s, cA), np.allclose(-d/np.sqrt(2), cD))
print(np.allclose(haar_lifting_odwrotny(s, d), x))

# wersja całkowitoliczbowa: dzielenie z zaokrągleniem w dół
def haar_calkowity(x):
    e, o = x[0::2], x[1::2]
    d = o - e
    return e + d//2, d

def haar_calkowity_odwrotny(s, d):
    e = s - d//2
    x = np.empty(2*len(s), dtype=s.dtype)
    x[0::2], x[1::2] = e, d + e
    return x

xi = np.array([1, 2, 2, 4, 3, 2, 1, 2, 2, 3])
si, di = haar_calkowity(xi)
print(si, di, np.array_equal(haar_calkowity_odwrotny(si, di), xi))

[1.5 3.  2.5 1.5 2.5] [ 1.  2. -1.  1.  1.]
True True
True
[1 3 2 1 2] [ 1  2 -1  1  1] True


### Kod 12.6. Falka 5/3 w schemacie liftingu i jej zgodność z falką bior2.2

In [7]:
import numpy as np
import pywt

def lifting_53(x):
    """Jeden stopień, sygnał traktowany jako okresowy."""
    e, o = x[0::2], x[1::2]
    d = o - (e + np.roll(e, -1))/2            # predykcja liniowa
    s = e + (np.roll(d, 1) + d)/4             # uaktualnienie
    return s, d

def lifting_53_calkowity(x):
    e, o = x[0::2], x[1::2]
    d = o - (e + np.roll(e, -1))//2           # wzór (12.7)
    s = e + (np.roll(d, 1) + d + 2)//4
    return s, d

def lifting_53_calkowity_odwrotny(s, d):
    e = s - (np.roll(d, 1) + d + 2)//4
    o = d + (e + np.roll(e, -1))//2
    x = np.empty(2*len(s), dtype=s.dtype)
    x[0::2], x[1::2] = e, o
    return x

rng = np.random.default_rng(0)
x = rng.integers(0, 256, size=64)

s, d = lifting_53(x.astype(float))
cA, cD = pywt.dwt(x.astype(float), "bior2.2", mode="periodization")
print(np.allclose(np.sqrt(2)*s, cA), np.allclose(-d/np.sqrt(2), cD))

si, di = lifting_53_calkowity(x)
print(si.dtype, np.array_equal(lifting_53_calkowity_odwrotny(si, di), x))

rampa = np.arange(0, 64, 2)                   # sygnał liniowy
print(lifting_53_calkowity(rampa)[1][:-1])    # detale poza brzegiem

True True
int64 True
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Ćwiczenia i rozwiązania

### Ćwiczenie 12.1
Dla danych z przykładu 12.1 wyznacz wartości na siatce o kroku 0,5 metodą interpolacji liniowej i funkcjami sklejanymi. W których punktach wyniki różnią się najbardziej?

### Rozwiązanie 12.1

In [8]:
from scipy.interpolate import CubicSpline
x = np.array([1., 2, 4, 5, 7, 8, 9]); y = np.array([1., 2, 6, 4, 2, 6, 5])
xn = np.arange(1, 9.01, 0.5)
yl, ys = np.interp(xn, x, y), CubicSpline(x, y)(xn)
for a, b, c in zip(xn, yl, ys):
    print(f"x = {a:3.1f}  liniowa {b:5.2f}  sklejana {c:5.2f}  różnica {c - b:6.2f}")
print("największa różnica w x =", xn[np.argmax(np.abs(ys - yl))])

x = 1.0  liniowa  1.00  sklejana  1.00  różnica   0.00
x = 1.5  liniowa  1.50  sklejana  1.09  różnica  -0.41
x = 2.0  liniowa  2.00  sklejana  2.00  różnica   0.00
x = 2.5  liniowa  3.00  sklejana  3.34  różnica   0.34
x = 3.0  liniowa  4.00  sklejana  4.71  różnica   0.71
x = 3.5  liniowa  5.00  sklejana  5.73  różnica   0.73
x = 4.0  liniowa  6.00  sklejana  6.00  różnica   0.00
x = 4.5  liniowa  5.00  sklejana  5.30  różnica   0.30
x = 5.0  liniowa  4.00  sklejana  4.00  różnica   0.00
x = 5.5  liniowa  3.50  sklejana  2.62  różnica  -0.88
x = 6.0  liniowa  3.00  sklejana  1.58  różnica  -1.42
x = 6.5  liniowa  2.50  sklejana  1.24  różnica  -1.26
x = 7.0  liniowa  2.00  sklejana  2.00  różnica   0.00
x = 7.5  liniowa  4.00  sklejana  3.94  różnica  -0.06
x = 8.0  liniowa  6.00  sklejana  6.00  różnica   0.00
x = 8.5  liniowa  5.50  sklejana  6.81  różnica   1.31
x = 9.0  liniowa  5.00  sklejana  5.00  różnica   0.00
największa różnica w x = 6.0


### Ćwiczenie 12.2
Pokaż, że zmniejszenie liczby próbek przez odrzucenie co drugiej próbki, bez filtrowania, powoduje aliasing. Użyj sygnału z kodu 12.2 próbkowanego 100 razy na jednostkę czasu.

### Rozwiązanie 12.2

In [9]:
from scipy.signal import decimate
fs = 100
t = np.arange(0, 4, 1/fs)
s = np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*40*t)
warianty = {"co druga próbka": s[::2], "z filtrem (decimate)": decimate(s, 2)}
for nazwa, y in warianty.items():
    widmo = np.abs(np.fft.rfft(y))/len(y)*2
    f = np.fft.rfftfreq(len(y), 2/fs)
    duze = f[widmo > 0.1]
    print(f"{nazwa}: składowe przy częstotliwościach {duze}")

co druga próbka: składowe przy częstotliwościach [ 5. 10.]
z filtrem (decimate): składowe przy częstotliwościach [5.]


### Ćwiczenie 12.3
Zbadaj działanie filtru Hampela z kodu 12.3 dla szerokości okna 3, 5 i 11 oraz progów 2, 3 i 4. Ile próbek jest niepotrzebnie zmienianych?

### Rozwiązanie 12.3

In [10]:
def filtr_hampela(y, okno=5, prog=3.0):
    y = y.copy(); zmienione = []
    for i in range(len(y)):
        otoczenie = y[max(0, i - okno):i + okno + 1]
        m = np.median(otoczenie); mad = np.median(np.abs(otoczenie - m))
        if np.abs(y[i] - m) > prog*1.4826*mad:
            y[i] = m; zmienione.append(i)
    return y, zmienione
rng = np.random.default_rng(4)
t = np.linspace(0, 1, 400)
y = np.sin(2*np.pi*2*t) + 0.5*t + 0.05*rng.normal(size=t.size)
bledy = [60, 150, 151, 300]
y[bledy] += [4, -3, 5, 6]
print("okno  próg  wykryte  niepotrzebnie zmienione")
for okno in (3, 5, 11):
    for prog in (2, 3, 4):
        _, zm = filtr_hampela(y, okno, prog)
        print(f"{okno:4d} {prog:5d} {len(set(zm) & set(bledy)):8d} {len(set(zm) - set(bledy)):8d}")

okno  próg  wykryte  niepotrzebnie zmienione
   3     2        4       23
   3     3        4        8
   3     4        4        5
   5     2        4       18
   5     3        4        4
   5     4        4        1
  11     2        4        6
  11     3        4        1
  11     4        4        0


### Ćwiczenie 12.4
Dodaj do gładkiego sygnału jedną wartość odstającą i porównaj energie poziomów dekompozycji przed jej usunięciem i po usunięciu.

### Rozwiązanie 12.4

In [11]:
t = np.linspace(0, 1, 1024, endpoint=False)
s = np.sin(2*np.pi*3*t)
z = s.copy(); z[500] += 10
for nazwa, y in (("bez błędu", s), ("z błędem", z)):
    wsp = pywt.wavedec(y, "db4", level=5, mode="periodization")
    print(f"{nazwa:10s}", " ".join(f"{np.sum(c**2):8.3f}" for c in wsp))

bez błędu   511.996    0.004    0.000    0.000    0.000    0.000
z błędem    519.018    5.474    8.504    6.784   27.887   48.715


### Ćwiczenie 12.5
Dla sygnału okresowego (całkowita liczba okresów sinusoidy) porównaj współczynniki przy brzegu w trybach `periodic` i `symmetric`. Który tryb jest teraz lepszy?

### Rozwiązanie 12.5

In [12]:
t = np.arange(1000)/1000
s = np.sin(2*np.pi*4*t + 0.7)
for tryb in ("periodic", "symmetric", "smooth"):
    cD = pywt.dwt(s, "db4", mode=tryb)[1]
    print(f"{tryb:10s} brzeg: {max(np.abs(cD[:4]).max(), np.abs(cD[-4:]).max()):.2e}"
          f"   środek: {np.abs(cD[8:-8]).max():.2e}")

periodic   brzeg: 1.51e-07   środek: 2.09e-07
symmetric  brzeg: 1.26e-03   środek: 2.09e-07
smooth     brzeg: 2.63e-05   środek: 2.09e-07


### Ćwiczenie 12.6
Wykonaj ręcznie jeden stopień liftingu Haara i liftingu 5/3 (wzory 12.5 i 12.6, przedłużenie okresowe) dla wektora (2, 4, 6, 8, 10, 12, 14, 16). Porównaj współczynniki detali.

### Rozwiązanie 12.6

In [13]:
x = np.arange(2., 17, 2)
e, o = x[0::2], x[1::2]
d_haar = o - e; s_haar = e + d_haar/2
d_53 = o - (e + np.roll(e, -1))/2; s_53 = e + (np.roll(d_53, 1) + d_53)/4
print("Haar: d =", d_haar, " s =", s_haar)
print("5/3:  d =", d_53, " s =", s_53)

Haar: d = [2. 2. 2. 2.]  s = [ 3.  7. 11. 15.]
5/3:  d = [0. 0. 0. 8.]  s = [ 4.  6. 10. 16.]


### Ćwiczenie 12.7
Zaimplementuj wielopoziomową całkowitoliczbową transformatę 5/3 i transformatę odwrotną. Sprawdź dokładną odwracalność dla losowego wektora liczb całkowitych o długości 256.

### Rozwiązanie 12.7

In [14]:
def krok(x):
    e, o = x[0::2], x[1::2]
    d = o - (e + np.roll(e, -1))//2
    return e + (np.roll(d, 1) + d + 2)//4, d

def krok_odwrotny(s, d):
    e = s - (np.roll(d, 1) + d + 2)//4
    x = np.empty(2*len(s), dtype=s.dtype)
    x[0::2], x[1::2] = e, d + (e + np.roll(e, -1))//2
    return x

def transformata_53(x, poziomy):
    detale = []
    for _ in range(poziomy):
        x, d = krok(x); detale.append(d)
    return x, detale

def odwrotna_53(s, detale):
    for d in reversed(detale):
        s = krok_odwrotny(s, d)
    return s

rng = np.random.default_rng(12)
x = rng.integers(0, 256, size=256)
s, detale = transformata_53(x, 5)
print([len(d) for d in detale], len(s), s.dtype)
print("dokładna odwracalność:", np.array_equal(odwrotna_53(s, detale), x))

[128, 64, 32, 16, 8] 8 int64
dokładna odwracalność: True


### Ćwiczenie 12.8
Oszacuj, ile bitów potrzeba do zapisu losowego wektora o wartościach 0–255 i do zapisu gładkiego sygnału o tych samych wartościach po całkowitoliczbowej transformacie 5/3. Jako miarę przyjmij entropię rozkładu wartości.

### Rozwiązanie 12.8

In [15]:
def entropia_bitow(v):
    """Entropia Shannona rozkładu wartości, w bitach na próbkę."""
    _, licznosci = np.unique(v, return_counts=True)
    p = licznosci/licznosci.sum()
    return float(-np.sum(p*np.log2(p)))

n = np.arange(4096)
sygnaly = {
    "losowy": np.random.default_rng(5).integers(0, 256, size=4096),
    "gładki": np.round(127.5 + 100*np.sin(2*np.pi*n/512)
                       + 25*np.sin(2*np.pi*n/97)).astype(int),
}
for nazwa, x in sygnaly.items():
    s, detale = transformata_53(x, 5)
    po = np.concatenate([s] + detale)
    print(f"{nazwa}: przed {entropia_bitow(x):.2f} bitu na próbkę, "
          f"po transformacie {entropia_bitow(po):.2f}")
# Dla danych losowych transformata niczego nie zmienia (a nawet pogarsza),
# dla sygnału gładkiego większość współczynników jest bliska zeru,
# więc do ich zapisu wystarcza znacznie mniej bitów.

losowy: przed 7.96 bitu na próbkę, po transformacie 8.37
gładki: przed 7.88 bitu na próbkę, po transformacie 2.13
